# 02. Error Analysis & Model Interpretability

**Project:** SW5 Satellite Collision Risk Predictor (NearMiss)  
**Goal:** Inspect false negatives (missed collisions), false positives (avoidable burns), and evaluate global SHAP explainability.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.load import load_config, load_parquet

cfg = load_config()
preds = load_parquet('data/processed/predictions.parquet')
hist = load_parquet('data/processed/cdm_history.parquet')
test_preds = preds[preds['split'] == 'test'].copy()
print(f'Loaded {len(test_preds)} locked test events.')


## 1. Locked Test Set Performance Comparison


In [2]:
from src.evaluate import compute_classification_metrics, compute_regression_metrics, compute_official_score

y_true = test_preds['actual_high_risk'].values
y_risk = test_preds['actual_final_log_risk'].values
y_prob = test_preds['high_risk_prob'].values
y_pred_risk = test_preds['pred_final_log_risk'].values

from src.evaluate import compute_classification_metrics, compute_regression_metrics, compute_official_style_score

base_binary = (test_preds['baseline_recommendation'] == 'MANOEUVRE').astype(int).values
model_binary = (test_preds['recommendation'] == 'MANOEUVRE').astype(int).values

base_cls = compute_classification_metrics(y_true, base_binary, y_score=test_preds['latest_log_risk'].values)
base_reg = compute_regression_metrics(y_risk, test_preds['latest_log_risk'].values, high_risk_threshold=-6.0)
base_score = compute_official_style_score(base_reg['mse_high_risk'], base_cls['f2'])

model_cls = compute_classification_metrics(y_true, model_binary, y_score=y_prob)
model_reg = compute_regression_metrics(y_risk, y_pred_risk, high_risk_threshold=-6.0)
model_score = compute_official_style_score(model_reg['mse_high_risk'], model_cls['f2'])

res_df = pd.DataFrame([
    {'Model': 'Baseline A (Persistence)', 'Recall': f"{base_cls['recall']:.2%}", 'Precision': f"{base_cls['precision']:.2%}", 'F2': base_cls['f2'], 'PR-AUC': base_cls['pr_auc'], 'Missed (FN)': base_cls['fn'], 'False Alarms (FP)': base_cls['fp'], 'Kelvins Score': base_score},
    {'Model': 'NearMiss Champion (RF+Platt)', 'Recall': f"{model_cls['recall']:.2%}", 'Precision': f"{model_cls['precision']:.2%}", 'F2': model_cls['f2'], 'PR-AUC': model_cls['pr_auc'], 'Missed (FN)': model_cls['fn'], 'False Alarms (FP)': model_cls['fp'], 'Kelvins Score': model_score},
])
res_df


## 2. False Negatives (Missed Dangerous Events)
Investigating events where ground truth is high risk ($y \ge -6.0$), but the model recommended `NO_MANOEUVRE`.


In [3]:
fn_events = test_preds[(test_preds['actual_high_risk'] == 1) & (test_preds['recommendation'] == 'NO_MANOEUVRE')]
print(f'Total False Negatives on Locked Test: {len(fn_events)}')
fn_events[['event_id', 'latest_log_risk', 'actual_final_log_risk', 'high_risk_prob', 'recommendation', 'confidence']].head(10)


## 3. Avoided Unnecessary Burns (Saved Maneuvers)
Events where Baseline A raised a false alarm, but NearMiss correctly recommended `NO_MANOEUVRE`.


In [4]:
saved = test_preds[(test_preds['baseline_recommendation'] == 'MANOEUVRE') & (test_preds['recommendation'] == 'NO_MANOEUVRE') & (test_preds['actual_high_risk'] == 0)]
print(f'Total Avoided Unnecessary Burns: {len(saved)}')
saved[['event_id', 'latest_log_risk', 'actual_final_log_risk', 'high_risk_prob', 'confidence']].head(10)


## 4. Global SHAP Feature Importances


In [5]:
from IPython.display import Image, display
display(Image(filename='reports/figures/shap_importance.png'))
display(Image(filename='reports/figures/shap_summary_beeswarm.png'))
